In [ ]:
import json
import re
from itertools import chain
from pathlib import Path

In [ ]:
def rgb_convert(rgb_parentheses):
    match = re.match(r'^rgb\( *(\d+), *(\d+), *(\d+) *\)$', rgb_parentheses)
    return f'{match[1]:>3} {match[2]:>3} {match[3]:>3}'

rgb_convert('rgb(0,109,44)')

In [ ]:
def color_sets_generator(name, data):
    keys = sorted(int(k) for k in data.keys() if re.match(r'^\d+$', k))
    max_columns = max(keys)
    for k in keys:
        colors = data[str(k)]
        yield [
            f'{rgb_convert(color)}\t{name} {i + 1}/{len(colors)}'
            for i, color in enumerate(colors)
        ] + (
            [ '  0   0   0\tEmpty' ] * (max_columns - len(colors))
        )

In [ ]:
def gimp_palette_from_colorbrewer(name, data):
    full_name = f'ColorBrewer - {data['properties']['type']} - {name}'
    file_name = full_name.replace(' ', '') + '.gpl'
    pal_type = data['properties']['type'].lower()

    color_sets = list(color_sets_generator(name, data))
    columns = len(color_sets[0])
    
    return file_name, '\n'.join([
        'GIMP Palette',
        f'Name: {full_name}',
        f'Columns: {columns}',
        f'# {len(color_sets)} color sets available, up to {columns} colors',
        f'# https://colorbrewer2.org/#type={pal_type}&scheme={name}&n=3',
        '# https://github.com/axismaps/colorbrewer/blob/master/colorbrewer_schemes.js',
        *chain.from_iterable(color_sets)
    ])

In [ ]:
BASEDIR = Path('./palettes/')

# This JSON file was manually written by slightly reformatting the original JS file:
# https://github.com/axismaps/colorbrewer/blob/master/colorbrewer_schemes.js
with open('colorbrewer_schemes.json') as f:
    data = json.load(f)
    for name, obj in data.items():
        file_name, palette = gimp_palette_from_colorbrewer(name, obj)
        (BASEDIR / file_name).write_text(palette)